К полю "доход_расход" нужно подтянуть "вид_дохода_расхода" из Справочника "Меппинг_опу". По значениям вид_дохода_расхода создавать доп столбцы из значений Субконто.

Смотрим 91.01:

1. Переоценка ГП - доп столбцов выделять не нужно.
2. Премии покупателям - для 91.01 это скорее премии от поставщиков за выполнение планового объема закупок, доп столбцы не нужны (КА уже есть), но нужно определять вид связи и сегмент.
3. Аренда - доп столбцов выделять не нужно.
4. Изменение ППА - добавить доп признак (столбец "объект для изм ппа") для Изменения ППА - столбец с ППА, чтобы найти по нему КА из Справочника (лист "ППА")
5. Финансовый доход - доп столбцов выделять не нужно.
6. Продажа активов - вид_доход_расход "Продажа прочего имущества" и "Продажа основных средств" свернуть 91.01 и 91.02 по номеру операции

In [1]:
import pandas as pd
import numpy as np
pd.options.display.float_format = '{:,.2f}'.format

In [3]:
intermediate_data_df = pd.read_parquet('intermediate_data.parquet')

In [5]:
# Справочник Меппинг ОПУ
reference_df = pd.read_excel('C:/Users/a.karabedyan/Documents/PythonProject/decryption_collector_refactored_open_code/_REFERENCE_DATA/Справочники.xlsx', sheet_name='Меппинг_опу')

# Справочник ППА
reference_ppa_df = pd.read_excel('C:/Users/a.karabedyan/Documents/PythonProject/decryption_collector_refactored_open_code/_REFERENCE_DATA/Справочники.xlsx', sheet_name='ППА')

In [7]:
df_91_all = intermediate_data_df.loc[intermediate_data_df['Имя_файла'].str.contains('_91.0', na=False)]
df_9101 = df_91_all.loc[df_91_all['Кт'].str.startswith('91.01')]
df_9102 = df_91_all.loc[df_91_all['Дт'].str.startswith('91.02')]

In [9]:
df_9101 = df_9101.rename(columns={'Субконто Кт_1': 'доход_расход', 'Сумма': 'оборот, тыс.ед.', 'Дт': 'Корр.счет', 'Кт': 'счет'})
CONTRACTOR_ACCOUNTS = ('58', '60', '62', '63', '66', '67', '75', '76')
PPA_ACCOUNTS = ('01.09', '02.01', '01.01')
mask = df_9101['Корр.счет'].astype(str).str.startswith(CONTRACTOR_ACCOUNTS, na=False)
df_9101['контрагент'] = df_9101['Субконто Дт_1'].where(mask, 'не_указано').astype('string')
df_9101['оборот, тыс.ед.'] = df_9101.loc[:, 'оборот, тыс.ед.'] / -1000

# Создаём составной ключ в справочнике и в данных
reference_df['_key'] = reference_df['счет'].astype(str) + '_' + reference_df['доход_расход'].astype(str)
df_9101['_key'] = df_9101['счет'].astype(str) + '_' + df_9101['доход_расход'].astype(str)

# Маппинг
mapping = reference_df.drop_duplicates(subset='_key').set_index('_key')['вид_дохода_расхода']
df_9101['вид_дохода_расхода'] = df_9101['_key'].map(mapping).astype('string')

# Убираем временный ключ
df_9101.drop(columns='_key', inplace=True)
reference_df.drop(columns='_key', inplace=True)

# Заполняем отсутствующие значения
df_9101['вид_дохода_расхода'] = df_9101['вид_дохода_расхода'].fillna("не_указано")
df_9101 = df_9101.dropna(axis='columns', how='all')

# -------------------Добавим доп признак (столбец "объект для изм ппа") для Изменения ППА - столбец с ППА, чтобы найти по нему КА из Справочника (лист "ППА") -----------------№
mask_ppa = df_9101['Корр.счет'].astype(str).str.startswith(PPA_ACCOUNTS, na=False)
df_9101['объект для изм ппа'] = df_9101['Субконто Дт_1'].where(mask_ppa, 'не_указано').astype('string')

# 1. Маска по типу дохода (единое условие для обоих случаев)
mask_income = df_9101['вид_дохода_расхода'] == "Доходы от выбытия прав пользования активами, изменения условий договоров аренды"

# 2. Маски по корр.счету
mask_01_09 = df_9101['Корр.счет'].str.startswith("01.09", na=False)
mask_02_01_or_01_01 = df_9101['Корр.счет'].str.startswith(("02.01", "01.01"), na=False)

# 3. Маппинги из справочника ППА (с защитой от дубликатов)
mapping_ppa = (
    reference_ppa_df
    .drop_duplicates(subset='ос_ппа')
    .set_index('ос_ппа')['контрагент']
)
mapping_transfer = (
    reference_ppa_df
    .drop_duplicates(subset='ос_после_перехода_в_собственность')
    .set_index('ос_после_перехода_в_собственность')['контрагент']
)

# 4. Условие 1: 01.09 → поиск по "ос_ппа"
mask_1 = mask_income & mask_01_09
mapped_1 = df_9101['объект для изм ппа'].map(mapping_ppa)

count_1 = (mask_1 & mapped_1.notna()).sum()
df_9101['контрагент'] = np.where(
    mask_1 & mapped_1.notna(),   # заменить только там, где условие И найдено значение
    mapped_1,                     # новое значение
    df_9101['контрагент']         # иначе — оставить текущее
)

# 5. Условие 2: 02.01 или 01.01 → поиск по "ос_после_перехода_в_собственность"
mask_2 = mask_income & mask_02_01_or_01_01
mapped_2 = df_9101['объект для изм ппа'].map(mapping_transfer)

count_2 = (mask_2 & mapped_2.notna()).sum()
df_9101['контрагент'] = np.where(
    mask_2 & mapped_2.notna(),
    mapped_2,
    df_9101['контрагент']
)

# 6. Явное приведение к string (np.where часто возвращает object)
df_9101['контрагент'] = df_9101['контрагент'].astype('string')


df_9101.head()

,Имя_файла,Дата,Документ,Корр.счет,счет,"оборот, тыс.ед.",Содержание_1,Содержание_2,Субконто Дт_1,Субконто Дт_2,доход_расход,Субконто Дт_3,контрагент,вид_дохода_расхода,объект для изм ппа
112255,РЗК_отчпровод_91.01_2025_.txt,2025-01-01 23:59:59,Корректировка долга ТЗ000000047 от 01.01.2025 ...,60.01,91.01,-0.65,Оплата,<NA>,РТ-РЕГИСТРАТОР АО,Счет на оплату по прейскуранту,Доходы и расходы по результатам инвентаризации...,<NA>,РТ-РЕГИСТРАТОР АО,Доходы от инвентаризации,не_указано
112256,РЗК_отчпровод_91.01_2025_.txt,2025-01-01 23:59:59,Корректировка долга ТЗ000000062 от 01.01.2025 ...,60.02,91.01,-0.01,Оплата (аванс),<NA>,ЦОК АПК ФГБУ,Договор оказания услуг РЦ-К № 130 от 05.02.2020г.,Доходы и расходы по результатам инвентаризации СХ,<NA>,ЦОК АПК ФГБУ,Доходы от инвентаризации,не_указано
112257,РЗК_отчпровод_91.01_2025_.txt,2025-01-01 23:59:59,Оприходование товаров ТЦ000000010 от 01.01.202...,43,91.01,-4.54,Оприходование излишков,Количество,Пшеница озимая Урожай 2024 4 класс,Склад ГП ф. Тацинский,Доходы и расходы по результатам инвентаризации СХ,<NA>,не_указано,Доходы от инвентаризации,не_указано
112258,РЗК_отчпровод_91.01_2025_.txt,2025-01-01 23:59:59,Оприходование товаров ТЦ000000013 от 01.01.202...,43,91.01,-9.23,Оприходование излишков,Количество,Пшеница озимая Урожай 2024 4 класс,Склад ГП ф. Тацинский,Доходы и расходы по результатам инвентаризации СХ,<NA>,не_указано,Доходы от инвентаризации,не_указано
112259,РЗК_отчпровод_91.01_2025_.txt,2025-01-01 23:59:59,Оприходование товаров ТЗ000000003 от 01.01.202...,43,91.01,-5.28,Оприходование излишков,Количество,Пшеница озимая Урожай 2024 4 класс,"ПАО ""Миллеровский элеватор""",Доходы и расходы по результатам инвентаризации СХ,<NA>,не_указано,Доходы от инвентаризации,не_указано


In [14]:
mask_a = df_9101['доход_расход']=="Доходы и расходы от реализации материалов и иных активов"
mask_b = df_9101['Корр.счет']=="01.09"

df_9101[mask_a]

,Имя_файла,Дата,Документ,Корр.счет,счет,"оборот, тыс.ед.",Содержание_1,Содержание_2,Субконто Дт_1,Субконто Дт_2,доход_расход,Субконто Дт_3,контрагент,вид_дохода_расхода,объект для изм ппа
113235,РЗК_отчпровод_91.01_2025_.txt,2025-02-13 13:20:11,Реализация товаров и услуг ВЛ000000010 от 13.0...,62.01,91.01,-1.92,Реализация товаров,<NA>,Толмачев Алексей Юрьевич ИП,ДОГОВОР ПОСТАВКИ № 589/23Р от 05.07.2023 г.,Доходы и расходы от реализации материалов и ин...,<NA>,Толмачев Алексей Юрьевич ИП,Продажа прочего имущества,не_указано
113255,РЗК_отчпровод_91.01_2025_.txt,2025-02-14 23:59:59,Реализация товаров и услуг БЛ000000008 от 14.0...,62.01,91.01,-0.46,Реализация товаров,<NA>,Толмачев Алексей Юрьевич ИП,ДОГОВОР ПОСТАВКИ № 589/23Р от 05.07.2023 г.,Доходы и расходы от реализации материалов и ин...,<NA>,Толмачев Алексей Юрьевич ИП,Продажа прочего имущества,не_указано
113256,РЗК_отчпровод_91.01_2025_.txt,2025-02-14 23:59:59,Реализация товаров и услуг ВЛ000000011 от 14.0...,62.01,91.01,-0.22,Реализация товаров,<NA>,Толмачев Алексей Юрьевич ИП,ДОГОВОР ПОСТАВКИ № 589/23Р от 05.07.2023 г.,Доходы и расходы от реализации материалов и ин...,<NA>,Толмачев Алексей Юрьевич ИП,Продажа прочего имущества,не_указано
113467,РЗК_отчпровод_91.01_2025_.txt,2025-02-26 23:59:59,Реализация товаров и услуг ТЦ000000052 от 26.0...,62.01,91.01,-34.40,Реализация товаров,<NA>,АКРОН СКРАП РОСТОВ ООО,ДОГОВОР № 215/24Р от 15.04.2024г. поставки лом...,Доходы и расходы от реализации материалов и ин...,<NA>,АКРОН СКРАП РОСТОВ ООО,Продажа прочего имущества,не_указано
113468,РЗК_отчпровод_91.01_2025_.txt,2025-02-26 23:59:59,Реализация товаров и услуг ТЦ000000053 от 26.0...,62.01,91.01,-243.95,Реализация товаров,<NA>,АКРОН СКРАП РОСТОВ ООО,ДОГОВОР № 215/24Р от 15.04.2024г. поставки лом...,Доходы и расходы от реализации материалов и ин...,<NA>,АКРОН СКРАП РОСТОВ ООО,Продажа прочего имущества,не_указано
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122175,РЗК_отчпровод_91.01_2025_.txt,2025-12-18 23:59:59,Реализация товаров и услуг ПР000000062 от 18.1...,62.01,91.01,-31.68,Реализация товаров,<NA>,ДОН-ПОЛИМЕРЮГ ООО,ДОГОВОР № 988/23Р от 21.11.2023,Доходы и расходы от реализации материалов и ин...,<NA>,ДОН-ПОЛИМЕРЮГ ООО,Продажа прочего имущества,не_указано
122231,РЗК_отчпровод_91.01_2025_.txt,2025-12-22 23:59:59,Реализация товаров и услуг ПР000000063 от 22.1...,62.01,91.01,-6.46,Реализация товаров,<NA>,ДОН-ПОЛИМЕРЮГ ООО,ДОГОВОР № 988/23Р от 21.11.2023,Доходы и расходы от реализации материалов и ин...,<NA>,ДОН-ПОЛИМЕРЮГ ООО,Продажа прочего имущества,не_указано
122232,РЗК_отчпровод_91.01_2025_.txt,2025-12-22 23:59:59,Реализация товаров и услуг ПР000000064 от 22.1...,62.01,91.01,-8.93,Реализация товаров,<NA>,ДОН-ПОЛИМЕРЮГ ООО,ДОГОВОР № 988/23Р от 21.11.2023,Доходы и расходы от реализации материалов и ин...,<NA>,ДОН-ПОЛИМЕРЮГ ООО,Продажа прочего имущества,не_указано
122249,РЗК_отчпровод_91.01_2025_.txt,2025-12-25 17:22:35,Реализация товаров и услуг ТЗ000000961 от 25.1...,62.01,91.01,-2.00,Реализация услуг,<NA>,ОХК УРАЛХИМ АО,Договор на выполнение научно-исследовательской...,Доходы и расходы от реализации материалов и ин...,<NA>,ОХК УРАЛХИМ АО,Продажа прочего имущества,не_указано


In [51]:
df_9102 = df_9102.rename(columns={'Субконто Дт_1': 'доход_расход', 'Сумма': 'оборот, тыс.ед.', 'Кт': 'Корр.счет', 'Дт': 'счет'})
mask = df_9102['Корр.счет'].astype(str).str.startswith(CONTRACTOR_ACCOUNTS, na=False)
df_9102['контрагент'] = df_9102['Субконто Кт_1'].where(mask, 'не_указано').astype('string')
df_9102['оборот, тыс.ед.'] = df_9102.loc[:, 'оборот, тыс.ед.'] / 1000
# df_9102_clean = df_9102.loc[:, ['счет', 'Документ', 'Корр.счет', 'доход_расход',  'контрагент', 'оборот, тыс.ед.']]
df_9102['вид_дохода_расхода'] = df_9102.loc[:, 'доход_расход'].map(mapping).fillna("не_указано")
df_9102.head()

,Имя_файла,Дата,Документ,счет,Корр.счет,"оборот, тыс.ед.",Содержание_1,Содержание_2,доход_расход,Субконто Дт_2,Субконто Кт_1,Субконто Кт_2,Субконто Кт_3,Субконто Дт_3,контрагент,вид_дохода_расхода
122435,РЗК_отчпровод_91.02_2025_.txt,2025-01-01,Авансовый отчет ВГ000000006 от 01.01.2025 0:00:00,91.02.1,71.01,39.71,Проведение новогоднего мероприятия по отчет бн...,<NA>,Доходы (расходы) НЕ НУ,<NA>,Камбулова Ирина Анатольевна,<NA>,<NA>,<NA>,не_указано,Прочие
122436,РЗК_отчпровод_91.02_2025_.txt,2025-01-01,Операция (бухгалтерский и налоговый учет) 0000...,91.02.1,10.11,-0.23,<NA>,Количество,Доходы и расходы по результатам инвентаризации...,<NA>,Перчатки х/б с 2-м латексным покрыт.,<NA>,<NA>,<NA>,не_указано,Доходы от инвентаризации
122437,РЗК_отчпровод_91.02_2025_.txt,2025-01-01,Операция (бухгалтерский и налоговый учет) 0000...,91.02.1,10.11,5.28,<NA>,Количество,Доходы и расходы по результатам инвентаризации...,<NA>,Перчатки латексные,<NA>,<NA>,<NA>,не_указано,Доходы от инвентаризации
122438,РЗК_отчпровод_91.02_2025_.txt,2025-01-01,Операция (бухгалтерский и налоговый учет) 0000...,91.02.1,10.05,0.00,<NA>,Количество,Доходы и расходы по результатам инвентаризации...,<NA>,Датчик контроля угла наклона ДУ-Р2 (беспроводной),<NA>,<NA>,<NA>,не_указано,Доходы от инвентаризации
122439,РЗК_отчпровод_91.02_2025_.txt,2025-01-01,Операция (бухгалтерский и налоговый учет) 0000...,91.02.1,10.05,-0.00,<NA>,Количество,Доходы и расходы по результатам инвентаризации...,<NA>,"Палец 26950030032/ 3507-01-80.00.002 (D=60мм, ...",<NA>,<NA>,<NA>,не_указано,Доходы от инвентаризации


In [79]:
df_final = pd.concat([df_9101, df_9102], ignore_index=True)
df_final

,Имя_файла,Дата,Документ,Корр.счет,счет,"оборот, тыс.ед.",Содержание_1,Содержание_2,Субконто Дт_1,Субконто Дт_2,доход_расход,Субконто Кт_2,Субконто Кт_3,Субконто Дт_3,контрагент,вид_дохода_расхода,Субконто Кт_1
0,РЗК_отчпровод_91.01_2025_.txt,2025-01-01 23:59:59,Корректировка долга ТЗ000000047 от 01.01.2025 ...,60.01,91.01,-0.65,Оплата,<NA>,РТ-РЕГИСТРАТОР АО,Счет на оплату по прейскуранту,Доходы и расходы по результатам инвентаризации...,<NA>,<NA>,<NA>,РТ-РЕГИСТРАТОР АО,Доходы от инвентаризации,<NA>
1,РЗК_отчпровод_91.01_2025_.txt,2025-01-01 23:59:59,Корректировка долга ТЗ000000062 от 01.01.2025 ...,60.02,91.01,-0.01,Оплата (аванс),<NA>,ЦОК АПК ФГБУ,Договор оказания услуг РЦ-К № 130 от 05.02.2020г.,Доходы и расходы по результатам инвентаризации СХ,<NA>,<NA>,<NA>,ЦОК АПК ФГБУ,Расходы по инвентаризации,<NA>
2,РЗК_отчпровод_91.01_2025_.txt,2025-01-01 23:59:59,Оприходование товаров ТЦ000000010 от 01.01.202...,43,91.01,-4.54,Оприходование излишков,Количество,Пшеница озимая Урожай 2024 4 класс,Склад ГП ф. Тацинский,Доходы и расходы по результатам инвентаризации СХ,<NA>,<NA>,<NA>,не_указано,Расходы по инвентаризации,<NA>
3,РЗК_отчпровод_91.01_2025_.txt,2025-01-01 23:59:59,Оприходование товаров ТЦ000000013 от 01.01.202...,43,91.01,-9.23,Оприходование излишков,Количество,Пшеница озимая Урожай 2024 4 класс,Склад ГП ф. Тацинский,Доходы и расходы по результатам инвентаризации СХ,<NA>,<NA>,<NA>,не_указано,Расходы по инвентаризации,<NA>
4,РЗК_отчпровод_91.01_2025_.txt,2025-01-01 23:59:59,Оприходование товаров ТЗ000000003 от 01.01.202...,43,91.01,-5.28,Оприходование излишков,Количество,Пшеница озимая Урожай 2024 4 класс,"ПАО ""Миллеровский элеватор""",Доходы и расходы по результатам инвентаризации СХ,<NA>,<NA>,<NA>,не_указано,Расходы по инвентаризации,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79133,РЗК_отчпровод_91.02_2025_.txt,2025-12-31 23:59:59,Списание расходов будущих периодов 00000000012...,97.21,91.02.1,311.54,Списание РБП,<NA>,<NA>,<NA>,Кредитное обслуживание и расходы по открытию к...,<NA>,<NA>,<NA>,не_указано,Кредитное обслуживание и расходы по открытию к...,ГАП Дополнительные расходы по КД №520B00ZQ6 от...
79134,РЗК_отчпровод_91.02_2025_.txt,2025-12-31 23:59:59,Списание расходов будущих периодов 00000000012...,97.21,91.02.1,124.09,Списание РБП,<NA>,<NA>,<NA>,Кредитное обслуживание и расходы по открытию к...,<NA>,<NA>,<NA>,не_указано,Кредитное обслуживание и расходы по открытию к...,ГАП Дополнительные расходы по КД №520B011642LZ...
79135,РЗК_отчпровод_91.02_2025_.txt,2025-12-31 23:59:59,Списание расходов будущих периодов 00000000012...,97.21,91.02.1,13.68,Списание РБП,<NA>,<NA>,<NA>,Кредитное обслуживание и расходы по открытию к...,<NA>,<NA>,<NA>,не_указано,Кредитное обслуживание и расходы по открытию к...,Дог.страх.№РД-35-04-0031875 от 29.11.2023 по д...
79136,РЗК_отчпровод_91.02_2025_.txt,2025-12-31 23:59:59,Списание расходов будущих периодов 00000000012...,97.21,91.02.1,27.27,Списание РБП,<NA>,<NA>,<NA>,Кредитное обслуживание и расходы по открытию к...,<NA>,<NA>,<NA>,не_указано,Кредитное обслуживание и расходы по открытию к...,Дог. страх. №80712/933/21354/25 от 09.01.25г п...


In [81]:
df_final = df_final.dropna(axis=1, how='all')
df_final

,Имя_файла,Дата,Документ,Корр.счет,счет,"оборот, тыс.ед.",Содержание_1,Содержание_2,Субконто Дт_1,Субконто Дт_2,доход_расход,Субконто Кт_2,Субконто Кт_3,Субконто Дт_3,контрагент,вид_дохода_расхода,Субконто Кт_1
0,РЗК_отчпровод_91.01_2025_.txt,2025-01-01 23:59:59,Корректировка долга ТЗ000000047 от 01.01.2025 ...,60.01,91.01,-0.65,Оплата,<NA>,РТ-РЕГИСТРАТОР АО,Счет на оплату по прейскуранту,Доходы и расходы по результатам инвентаризации...,<NA>,<NA>,<NA>,РТ-РЕГИСТРАТОР АО,Доходы от инвентаризации,<NA>
1,РЗК_отчпровод_91.01_2025_.txt,2025-01-01 23:59:59,Корректировка долга ТЗ000000062 от 01.01.2025 ...,60.02,91.01,-0.01,Оплата (аванс),<NA>,ЦОК АПК ФГБУ,Договор оказания услуг РЦ-К № 130 от 05.02.2020г.,Доходы и расходы по результатам инвентаризации СХ,<NA>,<NA>,<NA>,ЦОК АПК ФГБУ,Расходы по инвентаризации,<NA>
2,РЗК_отчпровод_91.01_2025_.txt,2025-01-01 23:59:59,Оприходование товаров ТЦ000000010 от 01.01.202...,43,91.01,-4.54,Оприходование излишков,Количество,Пшеница озимая Урожай 2024 4 класс,Склад ГП ф. Тацинский,Доходы и расходы по результатам инвентаризации СХ,<NA>,<NA>,<NA>,не_указано,Расходы по инвентаризации,<NA>
3,РЗК_отчпровод_91.01_2025_.txt,2025-01-01 23:59:59,Оприходование товаров ТЦ000000013 от 01.01.202...,43,91.01,-9.23,Оприходование излишков,Количество,Пшеница озимая Урожай 2024 4 класс,Склад ГП ф. Тацинский,Доходы и расходы по результатам инвентаризации СХ,<NA>,<NA>,<NA>,не_указано,Расходы по инвентаризации,<NA>
4,РЗК_отчпровод_91.01_2025_.txt,2025-01-01 23:59:59,Оприходование товаров ТЗ000000003 от 01.01.202...,43,91.01,-5.28,Оприходование излишков,Количество,Пшеница озимая Урожай 2024 4 класс,"ПАО ""Миллеровский элеватор""",Доходы и расходы по результатам инвентаризации СХ,<NA>,<NA>,<NA>,не_указано,Расходы по инвентаризации,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79133,РЗК_отчпровод_91.02_2025_.txt,2025-12-31 23:59:59,Списание расходов будущих периодов 00000000012...,97.21,91.02.1,311.54,Списание РБП,<NA>,<NA>,<NA>,Кредитное обслуживание и расходы по открытию к...,<NA>,<NA>,<NA>,не_указано,Кредитное обслуживание и расходы по открытию к...,ГАП Дополнительные расходы по КД №520B00ZQ6 от...
79134,РЗК_отчпровод_91.02_2025_.txt,2025-12-31 23:59:59,Списание расходов будущих периодов 00000000012...,97.21,91.02.1,124.09,Списание РБП,<NA>,<NA>,<NA>,Кредитное обслуживание и расходы по открытию к...,<NA>,<NA>,<NA>,не_указано,Кредитное обслуживание и расходы по открытию к...,ГАП Дополнительные расходы по КД №520B011642LZ...
79135,РЗК_отчпровод_91.02_2025_.txt,2025-12-31 23:59:59,Списание расходов будущих периодов 00000000012...,97.21,91.02.1,13.68,Списание РБП,<NA>,<NA>,<NA>,Кредитное обслуживание и расходы по открытию к...,<NA>,<NA>,<NA>,не_указано,Кредитное обслуживание и расходы по открытию к...,Дог.страх.№РД-35-04-0031875 от 29.11.2023 по д...
79136,РЗК_отчпровод_91.02_2025_.txt,2025-12-31 23:59:59,Списание расходов будущих периодов 00000000012...,97.21,91.02.1,27.27,Списание РБП,<NA>,<NA>,<NA>,Кредитное обслуживание и расходы по открытию к...,<NA>,<NA>,<NA>,не_указано,Кредитное обслуживание и расходы по открытию к...,Дог. страх. №80712/933/21354/25 от 09.01.25г п...


In [83]:
df_final['Субконто Кт_3'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 79138 entries, 0 to 79137
Series name: Субконто Кт_3
Non-Null Count  Dtype 
--------------  ----- 
5229 non-null   string
dtypes: string(1)
memory usage: 618.4 KB
